# 06a - Study 2: embeddings for the search encoder (Run 5, Arm 2)

Arm 2 ranks the same biographies with a search encoder that takes a task
instruction. This notebook only produces the embeddings, on a GPU, because the
step needs about 20 hours on the machine that runs notebook 06.

- **Question:** Does the ranking effect of the manipulation hold with a search
  encoder that takes a task instruction, instead of the general sentence
  encoder of Arm 1?
- **Expectation:** no new one; Arm 2 is a robustness arm for the ranking and
  carries the hypotheses of Arm 1.
- **Status:** second encoder of Run 5, fixed on 2026-10-11 before any result.
- **Unit and test:** cosine between a normalised advertisement embedding and a
  normalised biography embedding; the tests themselves run in notebooks 06 and
  06b.
- **Inputs:** `kaggle_bios.parquet` and `kaggle_jds.parquet` from the attached
  Kaggle dataset.
- **Outputs:** `arm2_bio_embeddings.parquet`, `arm2_jd_embeddings.parquet`,
  `arm2_metadata.json` in `/kaggle/working`; copy them to
  `outputs/study2_run5/arm2/` and run `scripts/resave_kaggle_parquet.py`.
- **Run:** on Kaggle with Accelerator GPU T4 x2 and Internet on, Save and Run
  All. About 10 min.
- **Decisions:** the encoder is `Qwen/Qwen3-Embedding-0.6B`, with
  `intfloat/e5-large-v2` as the substitute fixed in advance should it not load;
  the roles follow Wilson and Caliskan (2024), so the advertisement is the
  query and carries the instruction and the biography is the document. Job
  descriptions are embedded whole, `max_seq_length` is set from the longest
  text in the dataset, and the run stops if any text would be cut.

Order of the cells: settings, versions, inputs and context length, model, the
two embedding passes, and the metadata.


In [1]:
# Settings of the run. One comment per parameter.
from pathlib import Path
import json, os, time

MODEL_PRIMARY   = "Qwen/Qwen3-Embedding-0.6B"
MODEL_SUBSTITUTE = "intfloat/e5-large-v2"

# Prefixes of the substitute, used only if the primary model does not load.
SUBSTITUTE_QUERY_PREFIX    = "query: "
SUBSTITUTE_DOCUMENT_PREFIX = "passage: "

# Instruction of the model card, fixed in advance. The job description is the
# query; the biography is the document and gets no instruction.
INSTRUCTION = ("Instruct: Given a job description, retrieve professional "
               "biographies of candidates who fit the job\nQuery:{jd}")

SEQ_MARGIN      = 64        # tokens of headroom above the longest text
BATCH_BIOS      = 32
BATCH_JDS       = 2         # job descriptions are long
CHECKPOINT_EVERY = 2000
DTYPE           = "float32"  # embeddings are stored at this precision

OUT_DIR = Path("/kaggle/working")
BIO_FILE  = OUT_DIR / "arm2_bio_embeddings.parquet"
JD_FILE   = OUT_DIR / "arm2_jd_embeddings.parquet"
META_FILE = OUT_DIR / "arm2_metadata.json"

print(f"primary model    : {MODEL_PRIMARY}")
print(f"substitute model : {MODEL_SUBSTITUTE}")
print(f"stored dtype     : {DTYPE}")

# -- Input folder and GPU, checked before anything else ----------------------
REQUIRED_FILES = ["kaggle_bios.parquet", "kaggle_jds.parquet"]

def find_input_dir():
    # The folder under /kaggle/input that holds the input files.
    root = Path("/kaggle/input")
    if not root.exists():
        raise FileNotFoundError(
            "/kaggle/input does not exist. This notebook runs on Kaggle; attach "
            "the dataset with the input files first.")
    for candidate in sorted(root.rglob(REQUIRED_FILES[0])):
        folder = candidate.parent
        if all((folder / name).exists() for name in REQUIRED_FILES):
            return folder
    available = sorted(p.name for p in root.iterdir())
    raise FileNotFoundError(
        f"None of the attached datasets holds all of {REQUIRED_FILES}. "
        f"Attached: {available}")

IN_DIR = find_input_dir()
print(f"input folder     : {IN_DIR}")
for _name in REQUIRED_FILES:
    print(f"    {_name}  ({(IN_DIR / _name).stat().st_size / 1e6:.1f} MB)")

def require_gpu():
    # Report the accelerator and stop if there is no GPU.
    try:
        import torch
    except ImportError as error:
        raise RuntimeError(
            f"torch is not available, so the GPU cannot be checked: {error}") from error
    if torch.cuda.is_available():
        count = torch.cuda.device_count()
        names = [torch.cuda.get_device_name(i) for i in range(count)]
        memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU detected     : yes, {count} device(s): {', '.join(names)}")
        print(f"device memory    : {memory:.1f} GB")
        return True
    print("GPU detected     : no")
    raise RuntimeError(
        "No GPU is available. Embedding the Run 5 texts on a CPU would take about 20 hours. In the notebook settings choose "
        "Accelerator 'GPU T4 x2' and run again.")

require_gpu()

primary model    : Qwen/Qwen3-Embedding-0.6B
substitute model : intfloat/e5-large-v2
stored dtype     : float32
input folder     : /kaggle/input/datasets/norap99/study2-run5-llm-inputs
    kaggle_bios.parquet  (0.9 MB)
    kaggle_jds.parquet  (5.1 MB)
GPU detected     : yes, 2 device(s): Tesla T4, Tesla T4
device memory    : 15.6 GB


True

## 1. Versions

The model card of `Qwen/Qwen3-Embedding-0.6B` requires `transformers>=4.51.0` and
`sentence-transformers>=2.7.0`. The installed versions are printed; if they are
too old they are upgraded in place.

In [2]:
# The versions the model card asks for, upgraded in place if too old.
import importlib
from packaging import version as _version

def installed(name):
    try:
        return importlib.import_module(name).__version__
    except Exception:
        return None

needed = {"transformers": "4.51.0", "sentence_transformers": "2.7.0"}
print("installed before:")
for package, minimum in needed.items():
    print(f"  {package:<22} {installed(package)}  (needed {minimum} or newer)")

too_old = [p for p, m in needed.items()
           if installed(p) is None or _version.parse(installed(p)) < _version.parse(m)]
if too_old:
    names = {"sentence_transformers": "sentence-transformers"}
    targets = " ".join(f"'{names.get(p, p)}>={needed[p]}'" for p in too_old)
    print(f"\nupgrading: {targets}")
    !pip install -q -U {targets}
    for package in list(needed):
        importlib.invalidate_caches()
        module = importlib.import_module(package)
        importlib.reload(module)

import torch, transformers, sentence_transformers
print("\ninstalled now:")
print(f"  transformers          {transformers.__version__}")
print(f"  sentence-transformers {sentence_transformers.__version__}")
print(f"  torch                 {torch.__version__}")
print(f"  cuda available        {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  device                {torch.cuda.get_device_name(0)}")
    print(f"  device memory         {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

installed before:
  transformers           5.16.1  (needed 4.51.0 or newer)
  sentence_transformers  5.7.0  (needed 2.7.0 or newer)

installed now:
  transformers          5.16.1
  sentence-transformers 5.7.0
  torch                 2.11.0+cu128
  cuda available        True
  device                Tesla T4
  device memory         15.6 GB


## 2. Inputs

The same two text files that notebook 07a uses: every POOL biography in the
original form and in each arm that changed it, and every job description of the
Run 5 pools.

In [3]:
# Load the biography and job description texts.
import pandas as pd

bios = pd.read_parquet(IN_DIR / "kaggle_bios.parquet")
jds  = pd.read_parquet(IN_DIR / "kaggle_jds.parquet")

print(f"biography versions : {len(bios)}  {dict(bios.version.value_counts())}")
print(f"job descriptions   : {len(jds)}")
print(f"occupations        : {bios.occ.nunique()} / {jds.occ.nunique()}")
print(f"biography words    : median {bios.bio_words.median():.0f}  max {bios.bio_words.max()}")
print(f"job description words: median {jds.jd_words.median():.0f}  max {jds.jd_words.max()}")

# Every original biography of the POOL has to be present, plus the changed ones.
print(f"distinct biographies: {bios.bio_id.nunique()} (POOL is 3250)")

biography versions : 6221  {'original': np.int64(3250), 'communion': np.int64(1878), 'agency': np.int64(1093)}
job descriptions   : 2975
occupations        : 13 / 13
biography words    : median 53  max 148
job description words: median 388  max 2172
distinct biographies: 3250 (POOL is 3250)


## 3. Model

The primary model is loaded first. If that fails, the substitute fixed in
advance is used and the reason is printed. The exact revision and the dtype of
the weights are recorded, so the run can be repeated.

In [4]:
# Load the encoder and record model, revision and dtype.
from sentence_transformers import SentenceTransformer
from huggingface_hub import model_info

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = MODEL_PRIMARY
SUBSTITUTE_USED = False
SUBSTITUTE_REASON = ""

try:
    model = SentenceTransformer(MODEL_PRIMARY, device=DEVICE)
except Exception as error:                                   # noqa: BLE001
    SUBSTITUTE_USED = True
    SUBSTITUTE_REASON = f"{type(error).__name__}: {error}"
    MODEL_NAME = MODEL_SUBSTITUTE
    print(f"The primary model did not load, so the substitute fixed in advance "
          f"is used.\n  reason: {SUBSTITUTE_REASON}")
    model = SentenceTransformer(MODEL_SUBSTITUTE, device=DEVICE)

REVISION = ""
try:
    REVISION = model_info(MODEL_NAME).sha or ""
except Exception as error:                                   # noqa: BLE001
    print(f"revision could not be read from the hub: {error}")

parameters = sum(p.numel() for p in model.parameters())
weight_dtype = str(next(model.parameters()).dtype)
print(f"model            : {MODEL_NAME}")
print(f"revision         : {REVISION}")
print(f"substitute used  : {SUBSTITUTE_USED}")
print(f"parameters       : {parameters/1e6:.0f} M")
print(f"weight dtype     : {weight_dtype}")
print(f"device           : {DEVICE}")
print(f"embedding size   : {model.get_sentence_embedding_dimension()}")
print(f"max_seq_length   : {model.max_seq_length} (before adjustment)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

model            : Qwen/Qwen3-Embedding-0.6B
revision         : 97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3
substitute used  : False
parameters       : 596 M
weight dtype     : torch.bfloat16
device           : cuda
embedding size   : 1024
max_seq_length   : 32768 (before adjustment)


/tmp/ipykernel_49/3983053070.py:34: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"embedding size   : {model.get_sentence_embedding_dimension()}")


## 4. Context length

`max_seq_length` is set from the longest text that will be embedded, including
the instruction, so no job description is cut. The number of texts that would
still be truncated is printed and has to be zero.

In [5]:
# Set the sequence length from the longest text and stop if anything would be cut.
def as_query(jd_text):
    if SUBSTITUTE_USED:
        return SUBSTITUTE_QUERY_PREFIX + jd_text
    return INSTRUCTION.format(jd=jd_text)

def as_document(bio_text):
    if SUBSTITUTE_USED:
        return SUBSTITUTE_DOCUMENT_PREFIX + bio_text
    return bio_text

tokenizer = model.tokenizer
jd_lengths  = [len(tokenizer(as_query(t), add_special_tokens=True)["input_ids"])
               for t in jds.jd_text]
bio_lengths = [len(tokenizer(as_document(t), add_special_tokens=True)["input_ids"])
               for t in bios.bio_text]
longest = max(max(jd_lengths), max(bio_lengths))
print(f"longest job description: {max(jd_lengths)} tokens")
print(f"longest biography      : {max(bio_lengths)} tokens")
print(f"longest overall        : {longest} tokens")

model_limit = getattr(model[0].auto_model.config, "max_position_embeddings", longest + SEQ_MARGIN)
target = min(longest + SEQ_MARGIN, model_limit)
model.max_seq_length = target
print(f"model position limit   : {model_limit}")
print(f"max_seq_length set to  : {model.max_seq_length}")
n_truncated = sum(1 for n in jd_lengths + bio_lengths if n > model.max_seq_length)
print(f"texts that would be truncated (expected 0): {n_truncated}")
if n_truncated:
    raise RuntimeError(
        f"{n_truncated} texts are longer than max_seq_length; Part 2.5 requires "
        f"whole job descriptions without windowing.")

longest job description: 3230 tokens
longest biography      : 267 tokens
longest overall        : 3230 tokens
model position limit   : 32768
max_seq_length set to  : 3294
texts that would be truncated (expected 0): 0


## 5. Embedding

Normalised embeddings, so a dot product is the cosine. Biographies and job
descriptions are embedded separately because only the job descriptions carry the
instruction. Results are written to a checkpoint every 2000 texts.

In [6]:
# Embed biographies and job descriptions, with a checkpoint every 2000 texts.
import numpy as np

def embed(texts, batch_size, label):
    vectors = []
    started = time.time()
    for start in range(0, len(texts), CHECKPOINT_EVERY):
        chunk = texts[start:start + CHECKPOINT_EVERY]
        vectors.append(model.encode(chunk, batch_size=batch_size,
                                    normalize_embeddings=True,
                                    convert_to_numpy=True,
                                    show_progress_bar=False).astype(DTYPE))
        done = min(start + CHECKPOINT_EVERY, len(texts))
        elapsed = time.time() - started
        print(f"  {label}: {done:>5} / {len(texts)}  {elapsed/60:.1f} min  "
              f"({done/elapsed:.1f} texts/s)")
    return np.vstack(vectors)

print("biographies")
bio_vectors = embed([as_document(t) for t in bios.bio_text], BATCH_BIOS, "bios")
print("job descriptions")
jd_vectors = embed([as_query(t) for t in jds.jd_text], BATCH_JDS, "jds")
print(f"biography embeddings      : {bio_vectors.shape} {bio_vectors.dtype}")
print(f"job description embeddings: {jd_vectors.shape} {jd_vectors.dtype}")
norms = np.linalg.norm(bio_vectors, axis=1)
print(f"biography norms           : min {norms.min():.6f} max {norms.max():.6f} "
      f"(normalised, so 1)")

biographies
  bios:  2000 / 6221  1.4 min  (23.3 texts/s)
  bios:  4000 / 6221  2.9 min  (22.7 texts/s)
  bios:  6000 / 6221  4.5 min  (22.4 texts/s)
  bios:  6221 / 6221  4.6 min  (22.4 texts/s)
job descriptions
  jds:  2000 / 2975  12.1 min  (2.8 texts/s)
  jds:  2975 / 2975  18.2 min  (2.7 texts/s)
biography embeddings      : (6221, 1024) float32
job description embeddings: (2975, 1024) float32
biography norms           : min 0.998048 max 1.003804 (normalised, so 1)


## 6. Save outputs

In [7]:
# Write the embeddings and the metadata of the run.
bio_out = bios[["bio_key", "bio_id", "occ", "bio_idx", "version", "female"]].copy()
bio_out["embedding"] = list(bio_vectors)
bio_out.to_parquet(BIO_FILE, index=False)

jd_out = jds[["jd_id", "occ", "jd_idx"]].copy()
jd_out["embedding"] = list(jd_vectors)
jd_out.to_parquet(JD_FILE, index=False)

META_FILE.write_text(json.dumps({
    "model": MODEL_NAME,
    "revision": REVISION,
    "substitute_used": SUBSTITUTE_USED,
    "substitute_reason": SUBSTITUTE_REASON,
    "parameters": int(parameters),
    "weight_dtype": weight_dtype,
    "stored_dtype": DTYPE,
    "embedding_dim": int(model.get_sentence_embedding_dimension()),
    "max_seq_length": int(model.max_seq_length),
    "longest_text_tokens": int(longest),
    "n_truncated": int(n_truncated),
    "normalised": True,
    "instruction": INSTRUCTION,
    "substitute_query_prefix": SUBSTITUTE_QUERY_PREFIX,
    "substitute_document_prefix": SUBSTITUTE_DOCUMENT_PREFIX,
    "n_bios": int(len(bio_out)),
    "n_jds": int(len(jd_out)),
    "transformers": transformers.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "torch": torch.__version__,
    "device": DEVICE,
}, indent=2), encoding="utf-8")

for path in (BIO_FILE, JD_FILE, META_FILE):
    print(f"{path}  ({path.stat().st_size/1e6:.1f} MB)")

print()
print("Download the three files into outputs/study2_run5/arm2/ for notebook 06, "
      "then run 07a_llm_scoring_kaggle.ipynb.")

/kaggle/working/arm2_bio_embeddings.parquet  (9.7 MB)
/kaggle/working/arm2_jd_embeddings.parquet  (4.6 MB)
/kaggle/working/arm2_metadata.json  (0.0 MB)

Download the three files into outputs/study2_run5/arm2/ for notebook 06, then run 07a_llm_scoring_kaggle.ipynb.


/tmp/ipykernel_49/1066405061.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "embedding_dim": int(model.get_sentence_embedding_dimension()),
